# 03.P · Mini-project: Titanic survival, then ShopWave churn

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/03-classification/project/project.ipynb)

Read the [brief and rubric](README.md). Fill in every `TODO`. Run top to bottom before you look at the solution.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, roc_curve, precision_recall_curve, average_precision_score)

def score_table(models, X_test, y_test):
    """One row per model: accuracy, precision, recall, F1, ROC-AUC on the test set."""
    rows = []
    for name, m in models.items():
        p = m.predict(X_test)
        proba = m.predict_proba(X_test)[:, 1] if hasattr(m, "predict_proba") else p
        rows.append(dict(model=name, accuracy=accuracy_score(y_test, p), precision=precision_score(y_test, p, zero_division=0),
                         recall=recall_score(y_test, p), f1=f1_score(y_test, p), roc_auc=roc_auc_score(y_test, proba)))
    return pd.DataFrame(rows).set_index("model").round(3)

# Part A: Titanic

## A1. Load and handle missing values

In [ ]:
titanic = sns.load_dataset("titanic")
print(titanic.shape)
print(titanic.isna().sum()[lambda s: s > 0])
titanic.head()

In [ ]:
t = titanic.copy()
# TODO: fill age with the median and embarked with the most common value

## A2. Encode and split

In [ ]:
# TODO: sex -> 0/1, embarked -> 0/1/2; features pclass, sex, age, sibsp, parch, fare, embarked
# TODO: stratified 75/25 split -> Xt_train, Xt_test, yt_train, yt_test

## A3. Baselines and models

In [ ]:
class RuleModel:
    """A hand-written rule wrapped so score_table can treat it like a model."""
    def __init__(self, fn): self.fn = fn
    def fit(self, X, y): return self
    def predict(self, X): return np.asarray(self.fn(X)).astype(int)

models_t = {
    "nobody survives":   RuleModel(lambda X: np.zeros(len(X))),
    "all women survive": RuleModel(lambda X: X["sex"] == 1),
    # TODO: KNN (scaled), decision tree (max_depth=4), logistic regression (scaled)
}
for m in models_t.values():
    m.fit(Xt_train, yt_train)

## A4. Compare

In [ ]:
# TODO: score_table(models_t, Xt_test, yt_test)

## A5. What did the best model learn?

*TODO: one sentence, from the tree's first split or the logistic coefficients.*

# Part B: ShopWave churn

## B1. Build the customer table

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd, numpy as np
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail
from sklearn.model_selection import train_test_split

xlsx = ensure_online_retail()
csv = xlsx.with_name("online_retail_2010_2011.csv")          # fast cache written by lesson 01.3
raw = pd.read_csv(csv, parse_dates=["InvoiceDate"]) if csv.exists() else pd.read_excel(xlsx, sheet_name="Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

In [ ]:
# TODO: print the majority-class baseline accuracy

## B2. Four candidates

In [ ]:
models = {
    "rule: days_since_last > 90": RuleModel(lambda X: X["days_since_last"] > 90),
    # TODO: KNN (scaled, k=15), decision tree (max_depth=4), logistic regression (scaled)
}
for m in models.values():
    m.fit(X_train, y_train)

## B3. Comparison table

In [ ]:
# TODO

## B4. Curves for the best model by AUC

In [ ]:
# TODO: ROC curve and precision-recall curve, labelled

## B5. Choose the voucher threshold by cost

£5 per voucher sent, £60 per churner missed.

In [ ]:
def cost(y, proba, t, c_voucher=5, c_lost=60):
    pred = (proba >= t).astype(int)
    flagged = pred.sum(); missed = ((y == 1) & (pred == 0)).sum()
    return c_voucher * flagged + c_lost * missed

# TODO: scan thresholds 0..1, report the cheapest, and the cost at 0.5, at 0 (everyone) and at 1.01 (nobody)

## B6. Five insights for marketing

*TODO: five short sentences in business language.*

1.
2.
3.
4.
5.